In [2]:
import pandas as pd
import sqlite3
from sklearn.preprocessing import LabelEncoder

conexion = sqlite3.connect('../data/boca_juniors.db')
df = pd.read_sql_query("SELECT * FROM adn_boca", conexion)
conexion.close()

print(f"Dataset: {df.shape}")
print(df.isnull().sum())  # Verificamos nulos

Dataset: (386, 17)
nombre               0
temporada            0
posicion             0
edad                 0
partidos             0
goles                0
asistencias          0
pases_precisos       0
rating               0
etiqueta             0
goles_por_partido    0
asist_por_partido    0
rendimiento          0
participacion_gol    0
experiencia          0
pases_norm           0
perfil_ofensivo      0
dtype: int64


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 386 entries, 0 to 385
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   nombre             386 non-null    object 
 1   temporada          386 non-null    int64  
 2   posicion           386 non-null    object 
 3   edad               386 non-null    int64  
 4   partidos           386 non-null    int64  
 5   goles              386 non-null    int64  
 6   asistencias        386 non-null    int64  
 7   pases_precisos     386 non-null    float64
 8   rating             386 non-null    float64
 9   etiqueta           386 non-null    int64  
 10  goles_por_partido  386 non-null    float64
 11  asist_por_partido  386 non-null    float64
 12  rendimiento        386 non-null    float64
 13  participacion_gol  386 non-null    float64
 14  experiencia        386 non-null    int64  
 15  pases_norm         386 non-null    float64
 16  perfil_ofensivo    386 non

In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

FEATURES_CLUSTER = ["rating", "goles_por_partido", "rendimiento", "pases_norm"]
FEATURES_MODEL   = FEATURES_CLUSTER + ["asist_por_partido", "participacion_gol",
                                         "perfil_ofensivo", "edad", "partidos"]

# ── Paso 1: Clustering sin etiqueta ──
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURES_CLUSTER])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

# Mapeo manual al revisar centroides (ajustar según resultados)
print(df.groupby("cluster")[FEATURES_CLUSTER + ["etiqueta"]].mean().round(2))

# ── Paso 2: Clasificador con cluster como feature ──
X = df[FEATURES_MODEL + ["cluster"]]
y = df["etiqueta"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,          # limitado para evitar overfitting
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred,
      target_names=["No encaja", "Perfil Boca"]))

         rating  goles_por_partido  rendimiento  pases_norm  etiqueta
cluster                                                              
0          6.73               0.08         2.75        0.83      0.19
1          7.48               0.20         5.15        0.86      0.93
2          7.42               0.00         5.28        0.00      0.82
              precision    recall  f1-score   support

   No encaja       1.00      0.94      0.97        33
 Perfil Boca       0.96      1.00      0.98        45

    accuracy                           0.97        78
   macro avg       0.98      0.97      0.97        78
weighted avg       0.98      0.97      0.97        78



c:\Users\Agu\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\Agu\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:58:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
# Pruebas de hiperparametros
FEATURES_CLUSTER = ["rating", "goles_por_partido", "rendimiento", "pases_norm"]
FEATURES_MODEL   = FEATURES_CLUSTER + [
    "asist_por_partido", "participacion_gol",
    "perfil_ofensivo", "edad", "partidos"
]

# ── Paso 1: Clustering sin etiqueta ──
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURES_CLUSTER])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

print(df.groupby("cluster")[FEATURES_CLUSTER + ["etiqueta"]].mean().round(2))

# ── Paso 2: Clasificador ──
X = df[FEATURES_MODEL + ["cluster"]]
y = df["etiqueta"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    random_state=42
)

# activar early stopping para versiones viejas
model.set_params(early_stopping_rounds=20)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("Best iteration:", model.best_iteration)
y_pred = model.predict(X_test)
print(classification_report(
    y_test, y_pred,
    target_names=["No encaja", "Perfil Boca"]
))



c:\Users\Agu\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\Agu\anaconda3\Lib\site-packages\xgboost\callback.py:385: UserWarning: [17:58:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


         rating  goles_por_partido  rendimiento  pases_norm  etiqueta
cluster                                                              
0          6.73               0.08         2.75        0.83      0.19
1          7.48               0.20         5.15        0.86      0.93
2          7.42               0.00         5.28        0.00      0.82
Best iteration: 299
              precision    recall  f1-score   support

   No encaja       1.00      1.00      1.00        33
 Perfil Boca       1.00      1.00      1.00        45

    accuracy                           1.00        78
   macro avg       1.00      1.00      1.00        78
weighted avg       1.00      1.00      1.00        78



In [19]:
y_train_pred = model.predict(X_train)
print(classification_report(y_train, y_train_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       131
           1       1.00      1.00      1.00       177

    accuracy                           1.00       308
   macro avg       1.00      1.00      1.00       308
weighted avg       1.00      1.00      1.00       308



In [20]:
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(max_iter=500)
logreg.fit(X_train, y_train)
print(classification_report(y_test, logreg.predict(X_test)))


              precision    recall  f1-score   support

           0       0.88      0.85      0.86        33
           1       0.89      0.91      0.90        45

    accuracy                           0.88        78
   macro avg       0.88      0.88      0.88        78
weighted avg       0.88      0.88      0.88        78

